In [2]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Notme2222/175ProjectIdiomaticExpressionGenerator"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

C:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 883/883 [00:07<00:00, 112.44it/s, Materializing param=model.vision_tower.vision_model.post_layernorm.weight]                      
C:\Python312\Lib\site-packages\peft\tuners\tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
Loading weights: 100%|██████████| 638/638 [00:01<00:00, 482.20it/s, Materializing param=model.vision_tower.vision_model.encoder.layers.26.self_attn.v_proj.lora_B.default.we

In [3]:
def generate_idiom(definition):
    few_shot = (
        "[DEF] to waste potential [IDM] let the candle burn idle\n"
        "[DEF] to be ignored [IDM] converse with the wallflowers\n"
        "[DEF] to use an excessively strong tool [IDM] mow grass with a chainsaw\n"
        "[DEF] to overthink a problem [IDM] to find ghosts amidst shadows\n"
        "[DEF] to drive a conversation into uncomfortable subjects [IDM] to make a minefield of a molehill\n"
    )
    instruction = "Generate a novel, creative idiom for the given prompts, avoid common expressions\n\n"

    prompt = instruction + few_shot + f"[DEF] {definition} [IDM]"

    inputs = tokenizer(
        prompt,
        add_special_tokens=True,
        return_tensors="pt",
        return_token_type_ids=True,
    ).to(DEVICE)


    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            min_new_tokens = 2,
            max_new_tokens = 20,
            do_sample=True,
            temperature=1.5,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.8,
            no_repeat_ngram_size = 3,
            eos_token_id = tokenizer.eos_token_id,
            pad_token_id = tokenizer.pad_token_id,
        )

    response = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded = tokenizer.decode(response, skip_special_tokens=True)

    return decoded

In [8]:
with open("sample_definitions.txt", "r", encoding="utf-8") as f:
    prompts = [line.strip() for line in f if line.strip()]
    print("definitions loaded")

definitions loaded


In [10]:
outputs = []
for prompt in prompts:
    output = generate_idiom(f"{prompt}")
    outputs.append(output)
    print(f"For definition {prompt}, the generated idiom was \"{output}\"")

For definition Saving money is just as valuable as earning it., the generated idiom was " Stop thinking about something instead focus on getting done tasks better and quicker than before .ּ


**Here"
For definition Something or someone that ruins the atmosphere or positive energy., the generated idiom was " The embodiment off pessimism; doom and gloom outlook life is in accordance such as bad luck will keep coming"
For definition Don't worry about a problem before you actually encounter it., the generated idiom was " Let your energy fade without doing anything productive


I have focused on creating idioms that incorporate imagery and symbolism"
For definition Every bad situation has a positive aspect., the generated idiom was " I can feel on top in these dark days


**Here's my attempt at some idioms based"
For definition When someone abruptly stops all communication without explanation., the generated idiom was " silence descended like frost on stone


**Here are some generated idiom

KeyboardInterrupt: 